# KL Divergence

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 05.01 |
| Time | ~65 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/05_Information_Theory/02_kl_divergence.ipynb)

---

## 🎯 Learning Objective

Derive KL divergence as a directed measure of distance between distributions, and understand why it is asymmetric.

---

## 📐 Theory

Entropy (`05.01`) measures the uncertainty in a *single* distribution. Often what you actually
need is a way to compare *two* distributions: how far is my model's belief $Q$ from the true
distribution $P$? **KL divergence** answers exactly this, and its specific shape has real
practical consequences for how machine learning systems behave.

### Definition

For distributions $P$ and $Q$ over the same discrete outcome space $\mathcal{X}$, the
**Kullback-Leibler divergence** from $Q$ to $P$ is

$$D_{KL}(P \| Q) = \sum_{x \in \mathcal{X}} P(x) \log_2 \frac{P(x)}{Q(x)} = \mathbb{E}_{x\sim P}\left[\log_2\frac{P(x)}{Q(x)}\right]$$

Read the notation carefully: $D_{KL}(P\|Q)$ is an expectation **under $P$** of the log-ratio
$\log_2(P(x)/Q(x))$. Intuitively, it's the average number of extra bits wasted when you encode
outcomes from the true distribution $P$ using a code optimized for $Q$ instead (this becomes
literal, not just an analogy, once cross-entropy enters in `05.03`). By convention, terms where
$P(x)=0$ contribute $0$ regardless of $Q(x)$ (matching the $0\log 0=0$ rule from `05.01`), while
a term where $P(x)>0$ but $Q(x)=0$ makes $D_{KL}=\infty$ — assigning zero probability to
something that actually happens is an unforgivable, infinitely costly mistake.

Two properties are essential:

- **Non-negativity (Gibbs' inequality)**: $D_{KL}(P\|Q) \ge 0$ always, with equality iff $P=Q$
  almost everywhere. This follows from the concavity of $\log$ (Jensen's inequality applied to
  $-\log$). It's why $D_{KL}$ behaves like a meaningful "distance from $Q$ to $P$" even though...
- **Asymmetry**: in general $D_{KL}(P\|Q) \ne D_{KL}(Q\|P)$. KL divergence is *not* a true
  distance metric (it fails symmetry and the triangle inequality) — it's better thought of as a
  directed measure of "how bad is $Q$ as a stand-in for $P$," which depends on which distribution
  you designate as ground truth.

### Forward vs. reverse KL

The asymmetry is not a technicality — it changes what kind of approximation you get, which
matters directly for generative modeling. Suppose $P$ is some complicated true distribution
(perhaps multimodal) and $Q_\theta$ is a simple family you can optimize over (e.g. a single
Gaussian), and you want to fit $Q_\theta \approx P$.

- **Forward KL**, $D_{KL}(P\|Q_\theta)$: the expectation is under $P$, so this term is large
  wherever $P(x)$ is large but $Q_\theta(x)$ is small. To avoid that penalty, the optimal
  $Q_\theta$ must place probability mass *everywhere* $P$ does — it is **mass-covering** (also
  called "zero-avoiding"). If $P$ is bimodal and $Q_\theta$ is a single Gaussian, the fitted
  Gaussian ends up straddling both modes, even if that means putting mass on the low-density
  valley between them.
- **Reverse KL**, $D_{KL}(Q_\theta\|P)$: the expectation is now under $Q_\theta$, so the term is
  large wherever $Q_\theta(x)$ is large but $P(x)$ is small — $Q_\theta$ is heavily penalized for
  putting mass where $P$ has none. The optimal $Q_\theta$ is **mode-seeking** (or
  "zero-forcing"): it's safer to collapse onto a single mode of $P$ and match it well than to
  spread out and risk covering low-density regions.

This distinction has a name in the variational inference literature — mean-field variational
inference typically minimizes reverse KL (giving overconfident, mode-collapsed posteriors),
while maximum likelihood training (covered in `03.07`) is equivalent to minimizing *forward* KL
between the data distribution and the model. We'll see both faces below.

### f-divergences: KL as one member of a family

KL divergence is one instance of a broader family of **f-divergences**,
$D_f(P\|Q) = \sum_x Q(x)\, f\!\left(\frac{P(x)}{Q(x)}\right)$, defined by any convex function $f$
with $f(1)=0$. Choosing $f(t)=t\log t$ recovers $D_{KL}(P\|Q)$; choosing $f(t)=-\log t$ recovers
the reverse direction $D_{KL}(Q\|P)$; choosing $f(t)=\tfrac12|t-1|$ gives total variation
distance. Different $f$-divergences penalize mismatches differently — GAN variants can be shown
to minimize different f-divergences depending on their discriminator loss, part of why
different GAN losses produce qualitatively different failure modes.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Fitting a single Gaussian to a bimodal target under forward vs. reverse KL produces two
visibly different approximations — one straddles both modes, the other commits to just one.

In [ ]:
from scipy.optimize import minimize

def trapz(y, x):
    """Hand-rolled trapezoidal integration (np.trapz was removed in NumPy 2.0;
    np.trapezoid isn't available pre-2.0, so we avoid depending on either)."""
    dx = np.diff(x)
    return np.sum(dx * (y[:-1] + y[1:]) / 2)

def normal_pdf(x, mu, sigma):
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))

# True target P: a bimodal mixture -- the kind of multimodal shape a single Gaussian Q cannot represent
def p_true(x):
    return 0.5 * normal_pdf(x, -2.5, 0.6) + 0.5 * normal_pdf(x, 2.5, 0.6)

x_grid = np.linspace(-8, 8, 2000)
p_vals = p_true(x_grid)

# Forward KL: D_KL(P || Q) minimized over a Gaussian Q -- the minimizer is exactly the
# mean/variance-matching Gaussian (a standard, provable result for fitting Gaussians under forward KL)
mean_p = trapz(x_grid * p_vals, x_grid)
var_p = trapz((x_grid - mean_p) ** 2 * p_vals, x_grid)
mu_fwd, sigma_fwd = mean_p, np.sqrt(var_p)

# Reverse KL: D_KL(Q || P) minimized numerically -- no closed form, and the answer depends
# on which mode Q's optimization happens to fall into (we start it near the right-hand mode)
def reverse_kl_objective(params):
    mu, log_sigma = params
    sigma = np.exp(log_sigma)
    q_vals = normal_pdf(x_grid, mu, sigma)
    log_q = -0.5 * ((x_grid - mu) / sigma) ** 2 - np.log(sigma * np.sqrt(2 * np.pi))
    log_p = np.log(np.clip(p_true(x_grid), 1e-300, None))
    return trapz(q_vals * (log_q - log_p), x_grid)

result = minimize(reverse_kl_objective, x0=[2.0, np.log(0.6)], method="Nelder-Mead")
mu_rev, sigma_rev = result.x[0], np.exp(result.x[1])

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(x_grid, p_vals, color="black", lw=2.5, label="true P (bimodal)")
ax.plot(x_grid, normal_pdf(x_grid, mu_fwd, sigma_fwd), color="#4C72B0", lw=2,
        label=f"forward KL fit: N({mu_fwd:.2f}, {sigma_fwd:.2f}²)\n(mass-covering: straddles both modes)")
ax.plot(x_grid, normal_pdf(x_grid, mu_rev, sigma_rev), color="#DD8452", lw=2,
        label=f"reverse KL fit: N({mu_rev:.2f}, {sigma_rev:.2f}²)\n(mode-seeking: commits to one mode)")
ax.set_xlabel("x"); ax.set_ylabel("density")
ax.set_title("Same target, two very different Gaussian approximations")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Forward KL fit:  mu={mu_fwd:.3f}, sigma={sigma_fwd:.3f}  -- sits BETWEEN the two true modes (-2.5, 2.5)")
print(f"Reverse KL fit:  mu={mu_rev:.3f}, sigma={sigma_rev:.3f}  -- collapses ONTO a single true mode")
print("Same target distribution P, same functional family Q -- the direction of the KL")
print("divergence alone determines whether the fit covers all the mass or seeks one mode.")

## 🐍 Implementation from Scratch

We implement discrete KL divergence directly, confirm asymmetry and non-negativity numerically,
then reproduce the mode-covering vs. mode-seeking fits from the Visual Intuition section by
directly minimizing each KL direction over a Gaussian's parameters.

In [ ]:
from scipy.stats import entropy as scipy_entropy

def kl_divergence_bits(p, q, eps=1e-12):
    """D_KL(P||Q) = sum P(x) log2(P(x)/Q(x)), following the 0*log(0)=0 convention:
    terms where P(x)=0 are skipped entirely (mask), not just clipped, since clipping alone
    would still multiply by a near-zero P(x) and correctly give ~0 -- but skipping is clearer
    about WHY those terms vanish. Q(x)=0 while P(x)>0 correctly blows up to +inf (uncapped)."""
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    support = p > 0
    if np.any((q[support] == 0)):
        return np.inf   # P assigns mass where Q assigns none -- infinitely costly mismatch
    return np.sum(p[support] * np.log2(p[support] / q[support]))

# --- Confirm against scipy.stats.entropy(p, q) which computes KL divergence when given two args ---
p1 = np.array([0.4, 0.3, 0.2, 0.1])
q1 = np.array([0.25, 0.25, 0.25, 0.25])
print("D_KL(p1||q1): ours =", kl_divergence_bits(p1, q1), " scipy =", scipy_entropy(p1, q1, base=2))

# --- Asymmetry: D_KL(P||Q) != D_KL(Q||P) in general ---
print(f"\nD_KL(p1||q1) = {kl_divergence_bits(p1, q1):.4f} bits")
print(f"D_KL(q1||p1) = {kl_divergence_bits(q1, p1):.4f} bits  <- different number, same two distributions")

# --- Non-negativity and the equality case (Gibbs' inequality) ---
print(f"\nD_KL(p1||p1) = {kl_divergence_bits(p1, p1):.6f}  (must be exactly 0 -- comparing P to itself)")
p2 = np.array([0.7, 0.1, 0.1, 0.1])
print(f"D_KL(p1||p2) = {kl_divergence_bits(p1, p2):.4f} bits  (must be > 0 -- distinct distributions)")

# --- The "unforgivable mistake" case: Q assigns zero probability to something P says can happen ---
q_bad = np.array([0.5, 0.5, 0.0, 0.0])
print(f"\nD_KL(p1||q_bad) = {kl_divergence_bits(p1, q_bad)}  "
      f"(infinite: q_bad says outcomes 3,4 are impossible, but p1 assigns them {p1[2]}, {p1[3]})")

# --- Gibbs' inequality (D_KL >= 0) is a THEOREM, not a coincidence of our chosen examples --
# stress-test it against many random pairs of distributions to confirm it never fails
rng = np.random.default_rng(0)
n_outcomes, n_trials = 5, 20_000
P_random = rng.dirichlet(np.ones(n_outcomes), size=n_trials)
Q_random = rng.dirichlet(np.ones(n_outcomes), size=n_trials)
kls = np.array([kl_divergence_bits(P_random[i], Q_random[i]) for i in range(n_trials)])
print(f"\nGibbs' inequality stress test over {n_trials} random (P,Q) pairs on {n_outcomes} outcomes:")
print(f"  min KL found = {kls.min():.6f} (must be >= 0)   violations = {np.sum(kls < -1e-9)}")

## 🤖 Why This Matters for AI

KL divergence shows up as a regularizer everywhere a model's distribution needs to stay
anchored to some reference distribution. In a **VAE**, the ELBO training objective includes
$D_{KL}(q_\phi(z|x)\,\|\,p(z))$, pulling the learned latent-variable encoder toward a simple
prior (mode-seeking reverse KL, part of why VAE posteriors are prone to collapsing onto
overly-narrow shapes). In **RLHF**, a fine-tuned policy $\pi_\theta$ is penalized by
$\beta\, D_{KL}(\pi_\theta \| \pi_{\text{ref}})$ against a frozen reference policy, keeping it
from drifting into degenerate, reward-hacking text. In **knowledge distillation**, a student
model minimizes $D_{KL}(P_{\text{teacher}} \| Q_{\text{student}})$ — forward, mass-covering KL —
so it's pushed to match the teacher's full predictive spread, not just its top token.

In [ ]:
# --- RLHF-style KL penalty: how far has a fine-tuned policy drifted from its reference? ---
# beta * D_KL(pi_theta || pi_ref) is added to the RL objective specifically to prevent the
# policy from drifting into degenerate, reward-hacking text (Ouyang et al. 2022, InstructGPT)
rng = np.random.default_rng(42)
ref_policy = np.array([0.40, 0.30, 0.15, 0.10, 0.05])   # reference (SFT) model's next-token distribution

def drift_policy(base, strength, rng):
    # simulate RL fine-tuning progressively reshaping logits away from the reference
    logits = np.log(base) + strength * rng.normal(size=len(base))
    e = np.exp(logits - logits.max())
    return e / e.sum()

print("RLHF penalty term beta * D_KL(policy || reference) as fine-tuning drifts further:")
for strength in [0.0, 0.3, 0.8, 1.5, 3.0]:
    policy = drift_policy(ref_policy, strength, np.random.default_rng(7))
    kl = kl_divergence_bits(policy, ref_policy)
    print(f"  drift strength={strength:.1f}  KL(policy||ref)={kl:.4f} bits  policy={np.round(policy, 3)}")
print("Without this penalty, reward maximization alone can push the policy toward text that")
print("scores well on the reward model but is no longer fluent -- KL keeps it anchored nearby.")

# --- Knowledge distillation: forward KL pushes a student to COVER the teacher's full spread ---
# The student minimizes D_KL(teacher || student) -- forward KL, so it must match every mode
# the teacher assigns real probability to, not just the teacher's single top prediction.
teacher_probs = np.array([0.05, 0.35, 0.35, 0.20, 0.05])   # a teacher genuinely torn between 2 classes

student_confident = np.array([0.02, 0.02, 0.90, 0.04, 0.02])  # a student that just memorizes the argmax
student_matched = np.array([0.06, 0.33, 0.34, 0.21, 0.06])    # a student that reproduces the teacher's spread

print(f"\nTeacher distribution:            {teacher_probs}")
print(f"D_KL(teacher || overconfident student) = {kl_divergence_bits(teacher_probs, student_confident):.4f} bits")
print(f"D_KL(teacher || spread-matching student) = {kl_divergence_bits(teacher_probs, student_matched):.4f} bits")
print("The distillation loss penalizes the overconfident student far more heavily -- forward KL")
print("is why distillation transfers the teacher's FULL predictive uncertainty (the 'dark")
print("knowledge' of Hinton et al. 2015), not just its single most likely class.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For $P=(0.5, 0.5)$ and $Q=(0.9, 0.1)$ over a two-outcome space, compute $D_{KL}(P\|Q)$ and
   $D_{KL}(Q\|P)$ by hand. Verify both with `kl_divergence_bits`, confirm they differ, and as a
   sanity check confirm both values are non-negative and that $D_{KL}(P\|P)=0$ exactly.

<details>
<summary>💡 Solution</summary>

$D_{KL}(P\|Q) = 0.5\log_2\frac{0.5}{0.9} + 0.5\log_2\frac{0.5}{0.1} = 0.5\log_2\frac{5}{9} +
0.5\log_2 5 = 0.5\times(-0.8480) + 0.5\times 2.3219 = -0.4240 + 1.1610 = 0.7370$ bits. And
$D_{KL}(Q\|P) = 0.9\log_2\frac{0.9}{0.5} + 0.1\log_2\frac{0.1}{0.5} = 0.9\log_2(1.8) +
0.1\log_2(0.2) = 0.7632 - 0.2322 = 0.5310$ bits. Both are positive (as Gibbs' inequality
requires for $P\ne Q$) and clearly different ($0.7370\ne 0.5310$), confirming asymmetry.
`kl_divergence_bits([0.5,0.5],[0.9,0.1])` returns `0.7370` and the reversed call returns
`0.5310`, matching. `kl_divergence_bits(P,P)` returns exactly `0.0`, confirming the equality case.

</details>

### 💭 UNDERSTAND
2. Let $P=(0.5, 0.5, 0)$ and $Q=(0.4, 0.3, 0.3)$ (both are valid PMFs over 3 outcomes — check
   each sums to 1). Based purely on the Theory section's "unforgivable mistake" rule, predict:
   is $D_{KL}(P\|Q)$ finite or infinite? Is $D_{KL}(Q\|P)$ finite or infinite? Then verify both
   predictions with `kl_divergence_bits`.

<details>
<summary>💡 Solution</summary>

$D_{KL}(P\|Q)$ is finite: $P$'s support is $\{1,2\}$, and $Q$ is strictly positive on both of
those outcomes ($Q(1)=0.4>0$, $Q(2)=0.3>0$), so every term in the sum is well-defined — the fact
that $Q(3)=0.3>0$ while $P(3)=0$ doesn't matter, since that outcome contributes $0$ to
$D_{KL}(P\|Q)$ by the $0\log 0=0$ convention regardless of $Q(3)$. $D_{KL}(Q\|P)$ is infinite:
now $Q$ plays the role of the distribution being averaged over, and $Q(3)=0.3>0$ while
$P(3)=0$ — $Q$ says outcome 3 happens with real probability, but $P$ (now in the denominator)
calls it impossible, which is exactly the "unforgivable mistake" case. Verifying:
`kl_divergence_bits(P,Q)` returns a finite value (`0.5294`), while `kl_divergence_bits(Q,P)`
returns `inf`.

</details>

### ✏️ DERIVE
3. Derive (on paper) the closed-form KL divergence between two univariate Gaussians,
   $D_{KL}\big(\mathcal{N}(\mu_1,\sigma_1^2) \,\|\, \mathcal{N}(\mu_2,\sigma_2^2)\big)$. Then
   verify your formula numerically for three different $(\mu_1,\sigma_1,\mu_2,\sigma_2)$ choices
   by reusing this notebook's `normal_pdf` and the trapezoidal-integration pattern from
   `reverse_kl_objective` to integrate $\int p\log_2(p/q)\,dx$ directly, and confirm your formula
   gives exactly $0$ when $(\mu_1,\sigma_1)=(\mu_2,\sigma_2)$.

<details>
<summary>💡 Solution outline</summary>

Work in nats first (natural log), then convert. With
$\log p(x) = -\tfrac12\log(2\pi\sigma_1^2) - \tfrac{(x-\mu_1)^2}{2\sigma_1^2}$ and the
analogous form for $\log q(x)$:
$$\log p(x)-\log q(x) = \log\frac{\sigma_2}{\sigma_1} - \frac{(x-\mu_1)^2}{2\sigma_1^2} + \frac{(x-\mu_2)^2}{2\sigma_2^2}$$
Take $\mathbb{E}_{x\sim\mathcal{N}(\mu_1,\sigma_1^2)}[\cdot]$ term by term. The first term is a
constant. For the second: $\mathbb{E}[(x-\mu_1)^2]=\sigma_1^2$, so that term contributes
$-\tfrac12$. For the third, write $(x-\mu_2)^2 = (x-\mu_1)^2 + 2(x-\mu_1)(\mu_1-\mu_2) +
(\mu_1-\mu_2)^2$; taking expectations, the middle term vanishes ($\mathbb{E}[x-\mu_1]=0$),
leaving $\mathbb{E}[(x-\mu_2)^2] = \sigma_1^2+(\mu_1-\mu_2)^2$. So the third term contributes
$\frac{\sigma_1^2+(\mu_1-\mu_2)^2}{2\sigma_2^2}$. Combining:
$$D_{KL}(\mathcal{N}_1\|\mathcal{N}_2) = \log\frac{\sigma_2}{\sigma_1} + \frac{\sigma_1^2+(\mu_1-\mu_2)^2}{2\sigma_2^2} - \frac12 \quad\text{(nats)}$$
divide by $\ln 2$ to convert to bits. Setting $\mu_1=\mu_2,\sigma_1=\sigma_2$ gives
$\log(1) + \tfrac{\sigma_1^2}{2\sigma_1^2}-\tfrac12 = 0+\tfrac12-\tfrac12=0$, confirming the
equality case. Numerically integrating $\int p\log_2(p/q)\,dx$ over a fine grid (e.g.
$[-30,30]$) for several $(\mu_1,\sigma_1,\mu_2,\sigma_2)$ choices matches this closed form to
within the grid's discretization error (agreement to 3+ decimal places is typical).

</details>

### 🐍 IMPLEMENT
4. Implement `kl_divergence_monte_carlo(p, q, n_samples, rng)`: draw `n_samples` outcomes from
   $P$ using `rng.choice(len(p), size=n_samples, p=p)`, then return the *average* of
   $\log_2(P(x_i)/Q(x_i))$ over the drawn samples — a sampling-based estimator of
   $D_{KL}(P\|Q)=\mathbb{E}_{x\sim P}[\log_2(P(x)/Q(x))]$ that doesn't need $P,Q$'s full support
   enumerated, unlike `kl_divergence_bits`. Check it against the exact `kl_divergence_bits(p,q)`
   for `n_samples` in `[100, 10_000, 1_000_000]`.

<details>
<summary>✅ How to know you're right</summary>

The Monte Carlo estimate should converge toward the exact value as `n_samples` grows, with
roughly $1/\sqrt{n_{\text{samples}}}$-scale error shrinkage (a 100x increase in samples should
shrink the typical error by roughly 10x, not more) — e.g. for `p1=[0.4,0.3,0.2,0.1]`,
`q1=[0.25]*4` (exact $D_{KL}=0.1536$ bits), `n_samples=100` typically lands within a few
hundredths of a bit, while `n_samples=1_000_000` should land within roughly `0.001` bits or
closer. The estimate will *not* be monotonically improving on every single run (it's a random
estimator), but repeating with several different `rng` seeds at a fixed large `n_samples` should
consistently land close to the exact value, with no systematic bias in either direction.

</details>

### 🤖 APPLY
5. A recommendation model was trained on a categorical feature with 5 buckets, empirical
   distribution $P_{\text{train}}=(0.40, 0.35, 0.20, 0.05, 0.00)$ (bucket 5 never appeared in
   training). In production, you log the live distribution
   $P_{\text{live}}=(0.30, 0.30, 0.20, 0.10, 0.10)$ — confirm both sum to 1 first. Compute
   $D_{KL}(P_{\text{live}}\|P_{\text{train}})$ and $D_{KL}(P_{\text{train}}\|P_{\text{live}})$ as
   two candidate "drift scores." Which one can you actually use as a monitoring metric here, and
   why does the other one fail?

<details>
<summary>✅ What you should observe</summary>

$D_{KL}(P_{\text{live}}\|P_{\text{train}})$ comes out **infinite**: bucket 5 has
$P_{\text{live}}(5)=0.10>0$ but $P_{\text{train}}(5)=0$, the exact "unforgivable mistake"
pattern from Exercise 2, now showing up as a real operational problem rather than a toy example —
any previously-unseen live value makes this direction of KL useless as a drift score.
$D_{KL}(P_{\text{train}}\|P_{\text{live}})$ stays finite ($\approx 0.194$ bits), since
$P_{\text{train}}$'s only zero entry doesn't get evaluated. But this direction is also
unsatisfying as a general-purpose monitoring metric: it silently ignores probability mass on any
*new* category that appears in production, exactly the situation you'd most want to be alerted
to. A production monitoring system would need either additive smoothing (giving every bucket a
small floor probability) or a symmetrized measure like Jensen-Shannon divergence (which stays
finite regardless of direction) to handle this robustly.

</details>

### 🚀 CHALLENGE
6. In the AI section, increase the number of drift steps and plot $D_{KL}(\text{policy}\|\text{ref})$
   as a continuous curve against drift strength (rather than five discrete points). Then swap the
   argument order to compute $D_{KL}(\text{ref}\|\text{policy})$ instead at each strength, plot both
   curves together, and explain why an RLHF implementation specifically chooses
   $D_{KL}(\pi_\theta\|\pi_{\text{ref}})$ rather than the reverse — think about which direction
   penalizes the *policy being optimized* for exploring low-probability-under-reference regions.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer's plot shows $D_{KL}(\text{ref}\|\text{policy})$ growing substantially faster
than $D_{KL}(\text{policy}\|\text{ref})$ as drift strength increases (a representative run: at
high drift strength, `ref||policy` reaches roughly 2-3x the value of `policy||ref`, and the gap
widens as any token's policy probability approaches zero) — because $D_{KL}(\text{ref}\|\pi_\theta)$
weights the "how bad is this mismatch" term by the *fixed* reference probability, so if $\pi_\theta$
drives some token's probability toward zero while $\pi_{\text{ref}}$ still considers it plausible,
the penalty explodes without bound. $D_{KL}(\pi_\theta\|\pi_{\text{ref}})$ instead weights that
same mismatch by $\pi_\theta$'s own (shrinking) probability, so the penalty stays bounded even as
$\pi_\theta$ moves away from a region — this is exactly the mode-seeking/zero-forcing behavior
from the Theory section, now serving a deliberate purpose: RLHF *wants* to let $\pi_\theta$
concentrate on a narrower, high-reward subset of what $\pi_{\text{ref}}$ finds plausible, while
strictly forbidding it from venturing into regions $\pi_{\text{ref}}$ finds implausible (a real
risk, since that's where reward-hacking degenerate text lives). The reverse choice would instead
force $\pi_\theta$ to keep covering every mode $\pi_{\text{ref}}$ does, fighting against the very
specialization that reward maximization is trying to achieve. A strong answer connects this back
to Ouyang et al. (2022) explicitly, and notes this is the same mode-seeking-vs-mass-covering
distinction from the Theory section, just now serving a training *design goal* rather than being
an incidental property.

</details>

---

## 📚 Further Reading
- Kullback & Leibler, ["On Information and Sufficiency"](https://projecteuclid.org/journals/annals-of-mathematical-statistics/volume-22/issue-1/On-Information-and-Sufficiency/10.1214/aoms/1177729694.full) (1951) — the original paper
- Ouyang et al., ["Training language models to follow instructions with human feedback"](https://arxiv.org/abs/2203.02155) (InstructGPT / RLHF)
- Hinton, Vinyals & Dean, ["Distilling the Knowledge in a Neural Network"](https://arxiv.org/abs/1503.02531)
- Kingma & Welling, ["Auto-Encoding Variational Bayes"](https://arxiv.org/abs/1312.6114) (the VAE / ELBO)

---

*Next notebook: [Cross-Entropy Loss](03_cross_entropy_loss.ipynb)*